In [ ]:
import os
import cv2
import json
import numpy as np
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

def extract_and_prepare_dataset(root_dir=".", target_size=640, output_dir="Extracted_Dataset"):
    video_folder = os.path.join(root_dir, "Raw Videos")
    anno_folder = os.path.join(root_dir, "Annotations")
    
    # create folder for frame images
    img_dir = os.path.join(output_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    
    anno_files = sorted([f for f in os.listdir(anno_folder) if f.endswith('.xml')])
    master_labels = {}

    total_videos = len(anno_files)
    
    for v_idx, file in enumerate(anno_files):
        file_name = os.path.splitext(file)[0]
        anno_path = os.path.join(anno_folder, file)
        video_path = os.path.join(video_folder, f"{file_name}.mov")

        if not os.path.exists(video_path):
            continue

        # read the xml file to get boxes
        tree = ET.parse(anno_path)
        root_xml = tree.getroot()
        ball_map = {}
        
        for track in root_xml.findall('track'):
            if track.get('label') == 'baseball':

                # check for labels
                for box in track.findall('box'):
                    is_moving = any(attr.text == 'true' for attr in box.findall('attribute') if attr.get('name') == 'moving')
                    is_visible = box.get('outside') == '0'
                    
                    if is_moving and is_visible:
                        f_idx = int(box.get('frame'))
                        ball_map[f_idx] = [
                            float(box.get('xtl')), float(box.get('ytl')),
                            float(box.get('xbr')), float(box.get('ybr'))
                        ]

        # get video dimensions
        cap = cv2.VideoCapture(video_path)
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        std_res = max(h, w)
        scale_ratio = target_size / std_res

        # go through the video frame by frame
        for f in tqdm(range(total_frames), desc=f"Extracting Video {v_idx+1}/{total_videos}", leave=False):
            ret, frame = cap.read()
            if not ret: break
            
            # pad with black so it's a square, then resize it
            square = np.zeros((std_res, std_res, 3), dtype=np.uint8)
            square[0:h, 0:w] = frame
            frame_resized = cv2.resize(square, (target_size, target_size))
            
            # save the picture
            img_name = f"{file_name}_frame_{f:04d}.jpg"
            img_path = os.path.join(img_dir, img_name)
            cv2.imwrite(img_path, frame_resized)
            
            # adjust the box coordinates for the new image size
            if f in ball_map:
                raw_box = ball_map[f]
                scaled_box = [
                    max(0.0, min(target_size - 1.0, raw_box[0] * scale_ratio)),
                    max(0.0, min(target_size - 1.0, raw_box[1] * scale_ratio)),
                    max(0.0, min(target_size - 1.0, raw_box[2] * scale_ratio)),
                    max(0.0, min(target_size - 1.0, raw_box[3] * scale_ratio))
                ]
                
                # ignore glitched boxes
                if (scaled_box[2] - scaled_box[0]) >= 2.0 and (scaled_box[3] - scaled_box[1]) >= 2.0:
                    master_labels[img_name] = {"label": 1, "box": scaled_box}
                else:
                    master_labels[img_name] = {"label": 0, "box": None}
            else:
                # no moving ball in this frame
                master_labels[img_name] = {"label": 0, "box": None}
                
        cap.release()

    # save all labels into one json file
    json_path = os.path.join(output_dir, "master_labels.json")
    with open(json_path, "w") as f:
        json.dump(master_labels, f, indent=4)
# run the extractor
extract_and_prepare_dataset()

Found 61 videos. Starting extraction...


Extracting Video 1/61:   0%|          | 0/48 [00:00<?, ?it/s]

Extracting Video 2/61:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting Video 3/61:   0%|          | 0/51 [00:00<?, ?it/s]

Extracting Video 4/61:   0%|          | 0/50 [00:00<?, ?it/s]

...

Extracting Video 57/61:   0%|          | 0/102 [00:00<?, ?it/s]

Extracting Video 59/61:   0%|          | 0/76 [00:00<?, ?it/s]

Extracting Video 60/61:   0%|          | 0/57 [00:00<?, ?it/s]

Extracting Video 61/61:   0%|          | 0/62 [00:00<?, ?it/s]

Extraction Complete! 3634 images saved to Extracted_Dataset


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import os
import json

# prevent crash on empty frames
def collate_fn(batch):
    return tuple(zip(*batch))

class ExtractedBaseballDataset(Dataset):
    def __init__(self, dataset_dir="Extracted_Dataset"):
        self.img_dir = os.path.join(dataset_dir, "images")
        self.json_path = os.path.join(dataset_dir, "master_labels.json")
        
        # load list of coordinates
        with open(self.json_path, "r") as f:
            self.labels_dict = json.load(f)
            
        self.img_names = list(self.labels_dict.keys())
        
        # turn pictures into pytorch numbers
        self.transforms = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor() 
        ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # read the picture
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transforms(img)
        
        data = self.labels_dict[img_name]
        label = data["label"]
        box = data["box"]

        target = {}
        if label == 1 and box is not None:
            # ball is in the frame
            target["boxes"] = torch.tensor([box], dtype=torch.float32)
            target["labels"] = torch.tensor([1], dtype=torch.int64)
            target["area"] = torch.tensor([(box[2]-box[0]) * (box[3]-box[1])], dtype=torch.float32)
            target["iscrowd"] = torch.tensor([0], dtype=torch.int64)
        else:
            # empty frame setup
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
            target["labels"] = torch.zeros((0,), dtype=torch.int64)
            target["area"] = torch.zeros((0,), dtype=torch.float32)
            target["iscrowd"] = torch.zeros((0,), dtype=torch.int64)

        return img_tensor, target

def get_baseball_tracker_model():
    # load a pre-trained fasterrcnn
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    
    # change the final layer to look for just background and ball
    num_classes = 2 
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

In [3]:
import math
from tqdm.notebook import tqdm

def run_training():
    print("Loading dataset...")
    dataset = ExtractedBaseballDataset(dataset_dir="Extracted_Dataset")
    
    total_size = len(dataset)
    if total_size == 0:
        print("Error: No data found.")
        return

    # split train & test data
    train_size = int(0.8 * total_size)
    val_size = total_size - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # keep workers at 0 because freeze
    BATCH_SIZE = 32 
    WORKERS = 0
    
    print("Setting up data loaders...")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True, collate_fn=collate_fn)

    # use the graphics card if we have one
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Using device:", device)
    
    model = get_baseball_tracker_model().to(device)

    # load previous save
    save_file = "fasterrcnn_extracted.pth"
    if os.path.exists(save_file):
        print("Loading saved model...")
        model.load_state_dict(torch.load(save_file, map_location=device, weights_only=True))
    else:
        print("Starting from scratch!")
    
    # setup the optimizer (the tool that updates the model's weights)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=0.0001, weight_decay=0.0005)
    
    # prevent math errors when training
    scaler = torch.amp.GradScaler('cuda')

    num_epochs = 10

    print("Starting training loop...")
    for epoch in range(num_epochs):
        model.train() 
        epoch_loss = 0
        
        progress_bar = tqdm(train_loader, desc="Epoch " + str(epoch+1))
        
        for batch_imgs, batch_targets in progress_bar:
            
            # send to the graphics card
            images = list(img.to(device) for img in batch_imgs)
            targets = [{k: v.to(device) for k, v in t.items()} for t in batch_targets]

            optimizer.zero_grad()
            
            # mcalculate the loss
            with torch.amp.autocast('cuda'):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            
            # skip this batch if the math explodes
            if not math.isfinite(losses.item()):
                continue
            
            # update the model based on the loss
            scaler.scale(losses).backward()
            scaler.unscale_(optimizer) 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += losses.item()
            
            # simple progress bar update
            progress_bar.set_postfix(loss=round(losses.item(), 4))
            
        avg_loss = epoch_loss / len(train_loader)
        print("Epoch", epoch+1, ",Average loss:", round(avg_loss, 4))
        torch.save(model.state_dict(), save_file)

# Run training
run_training()

Loading dataset...
Setting up data loaders...
Using device: cuda
Loading saved model...
Starting training loop...


Epoch 1:   0%|          | 0/91 [00:00<?, ?it/s]

Epoch 1 ,Average loss: 0.0104


Epoch 2:   0%|          | 0/91 [00:00<?, ?it/s]

Epoch 2 ,Average loss: 0.0051


...

Epoch 6 ,Average loss: 0.0035


Epoch 7:   0%|          | 0/91 [00:00<?, ?it/s]

Epoch 7 ,Average loss: 0.0035


Epoch 8:   0%|          | 0/91 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import cv2
import torch
import numpy as np
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ToTensor
from tqdm.notebook import tqdm

# load the model setup
def get_baseball_tracker_model():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights=None)
    num_classes = 2 # 0: Background, 1: Baseball
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def test_model_on_video(input_video_path, output_video_path, model_weights_path="fasterrcnn_extracted.pth", conf_threshold=0.6):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Loading brain onto", device)
    
    # load the trained weights
    model = get_baseball_tracker_model().to(device)
    model.load_state_dict(torch.load(model_weights_path, map_location=device, weights_only=True))

    model.eval()
    
    # open the video we want to test
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error: Could not open video", input_video_path)
        return

    # get video meta data
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # save new video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # square padding
    std_res = max(height, width)
    target_size = 640
    scale_ratio = target_size / std_res

    print("Processing video frame by frame...")
    for _ in tqdm(range(total_frames), desc="Tracking Baseball"):
        ret, frame = cap.read()
        if not ret: break
        
        # format the frame
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        square = np.zeros((std_res, std_res, 3), dtype=np.uint8)
        square[0:height, 0:width] = frame_rgb
        resized = cv2.resize(square, (target_size, target_size))
        
        # get the picture ready 
        img_tensor = ToTensor()(resized).unsqueeze(0).to(device)
        
        # make predictions
        with torch.no_grad(): 
            predictions = model(img_tensor)
            
        prediction = predictions[0] 
        
        # pull out the coordinates and scores
        boxes = prediction['boxes'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()
        labels = prediction['labels'].cpu().numpy()
        
        for i in range(len(boxes)):
            # only draw a box if confidence is high
            if scores[i] >= conf_threshold and labels[i] == 1:
                box = boxes[i]
                
                # scale the 640x640 box back to the real video size
                x1 = int(box[0] / scale_ratio)
                y1 = int(box[1] / scale_ratio)
                x2 = int(box[2] / scale_ratio)
                y2 = int(box[3] / scale_ratio)
                
                # keep the box inside the screen
                x1, x2 = max(0, x1), min(width, x2)
                y1, y2 = max(0, y1), min(height, y2)
                
                # draw a green box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                
                # write the percentage above the box
                label_text = str(round(scores[i]*100, 1)) + "%"
                cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                
                # stop after drawing the best guess
                break 

        # save the drawn frame to new video
        out.write(frame)
        
    cap.release()
    out.release()
    print("Complete, "output_video_path, " saved")

# test model
input_video = "IMG_8225_amarnath.mov"
output_video = "annotated_result.mp4"

test_model_on_video(input_video, output_video)

Loading brain onto cuda
Processing video frame by frame...


Tracking Baseball:   0%|          | 0/48 [00:00<?, ?it/s]

Done! Check your folder for annotated_result.mp4
